In [1]:
import numpy as np
import pandas as pd

# sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

# model
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# 평가지표
from sklearn.metrics import accuracy_score

In [2]:
train = pd.read_csv('train.csv')
print(train.shape)
train.head(1)

(8693, 14)


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False


In [3]:
train.describe(include=['object', 'bool'])

C:\Users\user\AppData\Local\Temp\ipykernel_26908\2762557298.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  train.describe(include=['object', 'bool'])


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,VIP,Name,Transported
count,8693,8492,8476,8494,8511,8490,8493,8693
unique,8693,3,2,6560,3,2,8473,2
top,0001_01,Earth,False,G/734/S,TRAPPIST-1e,False,Elaney Webstephrey,True
freq,1,4602,5439,8,5915,8291,2,4378


In [4]:
X = train.drop(columns=['Name', 'Transported'])
X.shape

(8693, 12)

In [5]:
y = train['Transported']

In [6]:
# 파생변수 - 탐색보고서 참조 : PassengerId '_'문자열로 분리 후 반환값에 [0]번째만 새로운 'Group' 열로 추가
X['G'] = X['PassengerId'].str.split('_', expand=True)[0]

In [7]:
X.head(2)

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,G
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,0001
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,0002


In [8]:
X[['C1', 'C2', 'C3']] = X['Cabin'].str.split('/', expand=True)
X.head(2)

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,G,C1,C2,C3
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,0001,B,0,P
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,0002,F,0,S


In [9]:
# ISSUE
# X['G']에 숫자형태만 있나? Y
# -> 숫자형으로 변경? N

# X['C2']에 숫자형태만 있나? Y
# -> 숫자형으로 변경? Y
# -> nan값은 있나? Y

In [10]:
print(X['G'].isna().sum())
print(X['C2'].isna().sum())

0
199


In [11]:
X['G'].unique()

<StringArray>
['0001', '0002', '0003', '0004', '0005', '0006', '0007', '0008', '0009',
 '0010',
 ...
 '9267', '9268', '9270', '9272', '9274', '9275', '9276', '9278', '9279',
 '9280']
Length: 6217, dtype: str

In [12]:
X['C2'] = X['C2'].astype('Int64') # int 파이썬타입(결측치X), Int64 판다스타입(결측치)

In [13]:
# CryoSleep True/False bool인데 obejct로 되어있다. -> 결측치 'Missing'으로 변경
# 문자로 변경해서 통일
X['CryoSleep'] = X['CryoSleep'].astype('string')
X['CryoSleep'] = X['CryoSleep'].astype('object')

In [14]:
# VIP True/False bool인데 obejct로 되어있다. -> 결측치 'Missing'으로 변경
X['VIP'] = X['VIP'].astype('string')
X['VIP'] = X['VIP'].astype('object')

In [15]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   str    
 1   HomePlanet    8492 non-null   str    
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   str    
 4   Destination   8511 non-null   str    
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  G             8693 non-null   str    
 13  C1            8494 non-null   str    
 14  C2            8494 non-null   Int64  
 15  C3            8494 non-null   str    
dtypes: Int64(1), float64(6), object(2), str(7)
memory usage: 1.1+ MB


In [16]:
X = X.drop(columns=['PassengerId','Cabin'])

In [17]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   HomePlanet    8492 non-null   str    
 1   CryoSleep     8476 non-null   object 
 2   Destination   8511 non-null   str    
 3   Age           8514 non-null   float64
 4   VIP           8490 non-null   object 
 5   RoomService   8512 non-null   float64
 6   FoodCourt     8510 non-null   float64
 7   ShoppingMall  8485 non-null   float64
 8   Spa           8510 non-null   float64
 9   VRDeck        8505 non-null   float64
 10  G             8693 non-null   str    
 11  C1            8494 non-null   str    
 12  C2            8494 non-null   Int64  
 13  C3            8494 non-null   str    
dtypes: Int64(1), float64(6), object(2), str(5)
memory usage: 959.4+ KB


In [18]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [19]:
X_train.shape

(6954, 14)

In [20]:
cat_cols = X_train.select_dtypes('object').columns
num_cols = X_train.select_dtypes('number').columns
print(cat_cols)
print(num_cols)

Index(['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'G', 'C1', 'C3'], dtype='str')
Index(['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
       'C2'],
      dtype='str')


C:\Users\user\AppData\Local\Temp\ipykernel_26908\4071678993.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes('object').columns


In [21]:
print(X_train.isna().sum().sum())
print(X_val.isna().sum().sum())

2032
490


In [22]:
# 결측치
# 숫자 -> 각각의 열의 중앙값
# 범주 -> 모든 열에 'Missing'

a = X_train[num_cols].fillna(X_train[num_cols].median())
a2 = X_val[num_cols].fillna(X_val[num_cols].median())

X_train[num_cols] = a
X_val[num_cols] = a2

X_train[cat_cols] = X_train[cat_cols].fillna('Missing')
X_val[cat_cols] = X_val[cat_cols].fillna('Missing')

In [23]:
print(X_train.isna().sum().sum())
print(X_val.isna().sum().sum())

0
0


In [24]:
# 인코딩
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore').set_output(transform='pandas')

In [25]:
X_train_en = encoder.fit_transform(X_train[cat_cols])
X_val_en = encoder.transform(X_val[cat_cols])

In [26]:
# X_train의 수치형 컬럼 + X_train_en : pd.concat([df1, df2], axix=1)
# X_val의 수치형 컬럼 + X_val_en

In [27]:
X_train_pre = pd.concat([X_train[num_cols], X_train_en], axis=1)
X_val_pre = pd.concat([X_val[num_cols], X_val_en], axis=1)

In [28]:
print(X_train_pre.shape)
print(X_val_pre.shape)

(6954, 5260)
(1739, 5260)


In [29]:
# 학습

In [30]:
# model = RandomForestClassifier(n_estimators=300, random_state=42)
model = LogisticRegression(max_iter=300)

In [31]:
model.fit(X_train_pre, y_train)

d:\data_work\machine-learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 300 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=300).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",300
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in

In [32]:
y_pred = model.predict(X_val_pre)

accurracy = accuracy_score(y_val, y_pred)

print(accurracy)

0.7878090856814262


RandomForestClassifier : 0.7924094307073031

LogisticRegression(100번) 0.7860839562967222

0.7866589994249569